# Assignment 2 - Sistem Pencatatan Data Eksperimen Kecepatan Internet
**Nama:** Farid Al Farizi
**NIM:** 202410370110017

Program ini mencatat hasil eksperimen *speedtest* (download, upload, ping) milik setiap peneliti. Setiap pengguna hanya dapat melihat dan mengolah data eksperimen miliknya sendiri (*session lock*).

## 1. Struktur Data & Data Mock
| Data | Struktur | Alasan |
|---|---|---|
| Menu | `tuple` | Pilihan menu bersifat tetap, tidak boleh berubah saat program berjalan |
| Akun | `dict` (ID → password) | Pencarian kredensial langsung lewat key, ID otomatis unik |
| Profil | nested `dict` (ID → dict profil) | Satu ID terhubung ke banyak atribut profil |
| Data eksperimen | `list of dict` | Jumlah catatan dinamis (bisa bertambah/berkurang), urutan pengujian terjaga, dan setiap catatan punya field bernama dengan tipe beragam (str, float, int, bool) |

In [1]:
# (NIM: 202410370110017)

# Menu disimpan dalam tuple (fixed). Index 0-2 untuk tamu, index 3-8 untuk pengguna login
MAIN_MENU = (
    "Register",
    "Login",
    "Keluar Program",
    "Catat Hasil Speedtest",
    "Lihat Riwayat Speedtest",
    "Koreksi Hasil Speedtest",
    "Hapus Hasil Speedtest",
    "Ringkasan Kecepatan",
    "Logout",
)

# Account Dictionary: ID -> password
accounts = {
    "farid17": "fungsio26",
    "nayla02": "nayla123",
    "dika05": "dikapass",
}

# Profile Nested Dictionary: ID -> detail profil
profiles = {
    "farid17": {"name": "Farid", "phone": "081234567801", "role": "Peneliti Utama", "isp": "IndiHome"},
    "nayla02": {"name": "Nayla", "phone": "081234567802", "role": "Asisten Peneliti", "isp": "Biznet"},
    "dika05": {"name": "Dika", "phone": "081234567803", "role": "Asisten Peneliti", "isp": "Telkomsel"},
}

# Data eksperimen: List of Dictionaries, setiap catatan terikat ke "owner" (ID)
speed_tests = [
    {"test_id": 1, "owner": "farid17", "location": "Lab Informatika UMM", "connection": "WiFi",
     "download_mbps": 48.5, "upload_mbps": 20.3, "ping_ms": 14, "stable": True},
    {"test_id": 2, "owner": "nayla02", "location": "Kos Tlogomas", "connection": "WiFi",
     "download_mbps": 22.7, "upload_mbps": 8.1, "ping_ms": 31, "stable": False},
    {"test_id": 3, "owner": "dika05", "location": "Alun-Alun Batu", "connection": "Seluler",
     "download_mbps": 15.2, "upload_mbps": 5.6, "ping_ms": 45, "stable": True},
]

print("Data mock berhasil dimuat:", len(accounts), "akun,", len(speed_tests), "catatan eksperimen")

Data mock berhasil dimuat: 3 akun, 3 catatan eksperimen


## 2. Validasi Input (while loop + try-except)
Semua input pengguna melewati fungsi-fungsi ini. Input diulang terus sampai valid.

In [8]:
# (NIM: 202410370110017)

def input_text(prompt):
    #Meminta input teks dari user dan menolak input yang kosong.
    while True: # Looping terus sampai mendapat input yang valid
        try:
            value = input(prompt).strip() # Hapus spasi berlebih di awal/akhir
            if value == "":
                raise ValueError("Input tidak boleh kosong.")
        except ValueError as error:
            print(f"  [!] {error}") # Tampilkan pesan jika input kosong
        else:
            return value # Hentikan loop dan kembalikan teks valid


def input_number(prompt, number_type=float, min_value=0):
    #Meminta input angka dengan tipe dan batas minimal tertentu.
    while True:
        try:
            # Otomatis konversi teks jadi angka (float/int)
            value = number_type(input(prompt).strip()) 
            if value < min_value:
                raise ValueError(f"Nilai minimal {min_value}.")
        except ValueError as error:
            # Tangani error jika user mengetik huruf atau nilainya kurang dari minimal
            print(f"  [!] Input tidak valid. {error}") 
        else:
            return value # Hentikan loop dan kembalikan angka valid


def input_yes_no(prompt):
    #Meminta konfirmasi 'y' (Ya) atau 'n' (Tidak) dan mengembalikan nilai True/False.
    while True:
        try:
            value = input(prompt + " (y/n): ").strip().lower() # Paksa teks jadi huruf kecil
            if value not in ("y", "n"):
                raise ValueError("Ketik 'y' atau 'n'.")
        except ValueError as error:
            print(f"  [!] {error}")
        else:
            return value == "y" # Mengembalikan True jika "y", False jika "n"


def input_choice(prompt, options):
    #Meminta user mengetikkan angka yang terdaftar pada menu/options.
    while True:
        try:
            value = int(input(prompt).strip()) # Pastikan yang diketik adalah angka bulat (int)
            if value not in options:
                raise ValueError(f"Pilih salah satu dari {options}.")
        except ValueError as error:
            print(f"  [!] Pilihan tidak valid. {error}")
        else:
            return value # Hentikan loop dan kembalikan pilihan valid

## 3. Menu dengan Slicing

In [9]:
# (NIM: 202410370110017)

def get_menu(menu, is_logged_in):
    #Slicing tuple: tamu melihat menu[0:3], pengguna login melihat menu[3:].
    return menu[3:] if is_logged_in else menu[:3]


def show_menu(menu, is_logged_in, username=None):
    visible_menu = get_menu(menu, is_logged_in)
    title = f"MENU PENELITI ({username})" if is_logged_in else "MENU TAMU"
    print(f"\n===== {title} =====")
    for number, item in enumerate(visible_menu, start=1):
        print(f"{number}. {item}")
    return visible_menu

## 4. Registrasi & Login

In [ ]:
# (NIM: 202410370110017)

def register_user(account_data, profile_data):
    print("\n--- Registrasi Peneliti Baru ---")
    while True:
        try:
            user_id = input_text("ID baru       : ")
            if user_id in account_data:
                raise ValueError(f"ID '{user_id}' sudah digunakan, coba ID lain.")
        except ValueError as error:
            print(f"  [!] {error}")
        else:
            break

    password = input_text("Password      : ")
    name = input_text("Nama lengkap  : ")
    phone = input_text("No. telepon   : ")
    isp = input_text("Provider (ISP): ")

    account_data[user_id] = password
    profile_data[user_id] = {"name": name, "phone": phone, "role": "Peneliti", "isp": isp}
    print(f"  [OK] Registrasi berhasil. Silakan login sebagai '{user_id}'.")


def login_user(account_data, profile_data):
    print("\n--- Login ---")
    user_id = input_text("ID       : ")
    password = input_text("Password : ")
    if account_data.get(user_id) == password:
        print(f"  [OK] Selamat datang, {profile_data[user_id]['name']} ({profile_data[user_id]['role']})!")
        return user_id
    print("  [!] ID atau password salah.")
    return None

## 5. Manipulasi Data Eksperimen (Session Lock)
Setiap fungsi hanya memproses catatan yang `owner`-nya sama dengan ID yang sedang login.

In [11]:
# (NIM: 202410370110017)

def get_my_tests(test_data, username):
    #Session lock: ambil hanya catatan milik pengguna yang sedang login."""
    return [test for test in test_data if test["owner"] == username]


def find_my_test(test_data, username, test_id):
    for test in get_my_tests(test_data, username):
        if test["test_id"] == test_id:
            return test
    return None


def record_speedtest(test_data, username):
    print("\n--- Catat Hasil Speedtest ---")
    next_id = max([test["test_id"] for test in test_data], default=0) + 1
    new_test = {
        "test_id": next_id,
        "owner": username,
        "location": input_text("Lokasi pengujian      : "),
        "connection": input_text("Jenis koneksi (WiFi/Seluler/LAN): "),
        "download_mbps": input_number("Download (Mbps)       : ", float),
        "upload_mbps": input_number("Upload (Mbps)         : ", float),
        "ping_ms": input_number("Ping (ms)             : ", int),
        "stable": input_yes_no("Koneksi stabil?"),
    }
    test_data.append(new_test)
    print(f"  [OK] Hasil speedtest #{next_id} tersimpan.")


def show_speedtest_history(test_data, username):
    my_tests = get_my_tests(test_data, username)
    print(f"\n--- Riwayat Speedtest milik {username} ---")
    if not my_tests:
        print("  Belum ada catatan eksperimen.")
        return
    for test in my_tests:
        status = "Stabil" if test["stable"] else "Tidak stabil"
        print(f"  #{test['test_id']} | {test['location']} ({test['connection']}) | "
              f"DL {test['download_mbps']} Mbps | UL {test['upload_mbps']} Mbps | "
              f"Ping {test['ping_ms']} ms | {status}")


def correct_speedtest(test_data, username):
    show_speedtest_history(test_data, username)
    test_id = input_number("\nID pengujian yang dikoreksi: ", int, 1)
    test = find_my_test(test_data, username, test_id)
    if test is None:
        print("  [!] Catatan tidak ditemukan atau bukan milik Anda.")
        return
    test["download_mbps"] = input_number("Download baru (Mbps): ", float)
    test["upload_mbps"] = input_number("Upload baru (Mbps)  : ", float)
    test["ping_ms"] = input_number("Ping baru (ms)      : ", int)
    test["stable"] = input_yes_no("Koneksi stabil?")
    print(f"  [OK] Catatan #{test_id} berhasil dikoreksi.")


def discard_speedtest(test_data, username):
    show_speedtest_history(test_data, username)
    test_id = input_number("\nID pengujian yang dihapus: ", int, 1)
    test = find_my_test(test_data, username, test_id)
    if test is None:
        print("  [!] Catatan tidak ditemukan atau bukan milik Anda.")
        return
    test_data.remove(test)
    print(f"  [OK] Catatan #{test_id} dihapus.")


def summarize_speed(test_data, username):
    my_tests = get_my_tests(test_data, username)
    print(f"\n--- Ringkasan Kecepatan {username} ---")
    if not my_tests:
        print("  Belum ada data untuk diringkas.")
        return
    total = len(my_tests)
    avg_download = sum(test["download_mbps"] for test in my_tests) / total
    avg_upload = sum(test["upload_mbps"] for test in my_tests) / total
    avg_ping = sum(test["ping_ms"] for test in my_tests) / total
    fastest = max(my_tests, key=lambda test: test["download_mbps"])
    print(f"  Jumlah pengujian  : {total}")
    print(f"  Rata-rata download: {avg_download:.2f} Mbps")
    print(f"  Rata-rata upload  : {avg_upload:.2f} Mbps")
    print(f"  Rata-rata ping    : {avg_ping:.1f} ms")
    print(f"  Lokasi tercepat   : {fastest['location']} ({fastest['download_mbps']} Mbps)")

## 6. Program Utama
Jalankan sel ini untuk memulai program. Akun mock untuk mencoba: `farid17` / `fungsio26`.

In [ ]:
# (NIM: 202410370110017)

def main():
    current_user = None
    while True:
        is_logged_in = current_user is not None
        visible_menu = show_menu(MAIN_MENU, is_logged_in, current_user)
        choice = input_choice("Pilih menu: ", tuple(range(1, len(visible_menu) + 1)))
        selected = visible_menu[choice - 1]

        if selected == "Register":
            register_user(accounts, profiles)
        elif selected == "Login":
            current_user = login_user(accounts, profiles)
        elif selected == "Keluar Program":
            print("Program selesai. Terima kasih!")
            break
        elif selected == "Catat Hasil Speedtest":
            record_speedtest(speed_tests, current_user)
        elif selected == "Lihat Riwayat Speedtest":
            show_speedtest_history(speed_tests, current_user)
        elif selected == "Koreksi Hasil Speedtest":
            correct_speedtest(speed_tests, current_user)
        elif selected == "Hapus Hasil Speedtest":
            discard_speedtest(speed_tests, current_user)
        elif selected == "Ringkasan Kecepatan":
            summarize_speed(speed_tests, current_user)
        elif selected == "Logout":
            print(f"  [OK] {current_user} logout.")
            current_user = None


main()


===== MENU TAMU =====
1. Register
2. Login
3. Keluar Program
  [!] Pilihan tidak valid. invalid literal for int() with base 10: 'a'

--- Registrasi Peneliti Baru ---
  [!] ID 'farid17' sudah digunakan, coba ID lain.
